# Check if all datasets are organized well

Consequential structure/naming of files & folders
-hopefully already from fmriprep output

In [1]:
import os
import os.path as op
import pandas as pd

datapool_root_folder = '/mnt_AdaBD_largefiles/Data/DNumRisk_Data'#. #'/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk'
dataset =  'smile' #'numberline' #
bids_folder = op.join(datapool_root_folder, f'ds-{dataset}')

subList = [int(d[4:]) for d in os.listdir(bids_folder) if d.startswith('sub-')]
sesList = [1,2]

In [2]:
print(*subList, sep=' ')
len(subList)

101 102 103 104 105 106 107 108 109 110 111 114 116 117 118 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 302 303 305 306 307 308 310 311 312 314 315 316


43

## DataDim intermediate products

In [6]:
derivative = 'gradients' # correlation_matrices
derivative_obj_name = 'g-aligned' # '_funcCM' # 'lambdas_space-fsaverag5'#
sessions = '1-2'
tasks = 'digitorder' if dataset=='numberline' else 'magjudge-placevalue-rest'

missingSubs = []
for sub in subList:
    sub_folder = op.join(bids_folder,'derivatives',derivative,f'sub-{sub:03d}')
    fn = op.join(sub_folder, f'sub-{sub}_ses-{sessions}_task-{tasks}_{derivative_obj_name}.npy')
    if not op.exists(fn):
        missingSubs.append(sub)

print('Missing subjects:', missingSubs)
print('Completed :', len(subList) - len(missingSubs))

Missing subjects: []
Completed : 43


In [4]:
print(*missingSubs, sep=' ')    

127 320 327


In [5]:
derivative = 'gradients' # c'correlation_matrices' # 
derivative_obj_name = 'g-aligned' if derivative=='gradients' else 'funcCM' # 'lambdas_space-fsaverag5'#
sessions = '1-2'
tasks = 'rest' #  placevalue  rest  magjudge

missingSubs = []
for sub in subList:
    sub_folder = op.join(bids_folder,'derivatives',derivative,f'sub-{sub:03d}')
    fn = op.join(sub_folder, f'sub-{sub}_ses-{sessions}_task-{tasks}_{derivative_obj_name}.npy')
    if not op.exists(fn):
        missingSubs.append(sub)

print('Missing subjects:', missingSubs)
print('Completed :', len(subList) - len(missingSubs))

Missing subjects: [212, 213, 214, 215, 216, 302, 303, 305, 306, 307, 308, 310]
Completed : 31


In [6]:
print(*missingSubs, sep=' ')    

212 213 214 215 216 302 303 305 306 307 308 310


## Running parrallelized

In [ ]:
# parrellelizing running script (run via ipython in terminal)

import subprocess
from multiprocessing import Pool, cpu_count

subjects = subList[:10]#list(range(1, 67))  # e.g., subjects 1–20

bids_folder = "/mnt_AdaBD_largefiles/Data/SMILE_DATA/DNumRisk/ds-numberline"
sessions = '1-2'
tasks = 'digitorder'

def run_subject(sub):
    subprocess.run(["python", "fit_gradients_01.py", str(sub),
                    "--bids_folder", bids_folder,
                    "--sessions", sessions,
                    "--tasks", tasks]) # if ztransf 
    
n_cores = cpu_count() - 5  # leave five cores free

with Pool(processes=n_cores) as pool:
    pool.map(run_subject, subjects)

# pooling seems weird, early subjects dont work....

2.1

## Fmriprep

In [16]:
dataset = 'smile' #'numberline' # 
bids_folder = op.join(datapool_root_folder, f'ds-{dataset}')
subList = [int(d[4:]) for d in os.listdir(bids_folder) if d.startswith('sub-')]

fmriprep_root = op.join(bids_folder, "derivatives", "fmriprep")

import re
pattern = re.compile(
    r"sub-(?P<sub>[0-9]+)_"
    r"ses-(?P<ses>[0-9]+)_"
    r"task-(?P<task>[a-zA-Z0-9]+).*\.gii$"
)


rows = []

for sub in subList[:-10]:  # Exclude last 5 subjects for testing
    sub_id = sub
    sub_dir = op.join(fmriprep_root, f"sub-{sub}")

    # Find sessions for this subject
    sessions = [d for d in os.listdir(sub_dir) if d.startswith("ses-")]
    for ses in sorted(sessions):
        ses_id = ses.split("-")[1]

        func_dir = op.join(sub_dir, ses, "func")
        if not op.isdir(func_dir):
            continue
        
        # Scan functional files
        for f in os.listdir(func_dir):
            if not f.endswith("space-fsaverage5_hemi-L_bold.func.gii"):
                continue

            m = pattern.match(f)
            if m:
                task = m.group("task")
                rows.append((sub_id, ses_id, task, f))

# Build DataFrame
df = pd.DataFrame(rows, columns=["subject", "session", "task", "filename"])
df = df.set_index(["subject", "session", "task"]).sort_index()


In [19]:
df

filename
subject session task                                                         
101     1       magjudge    sub-101_ses-1_task-magjudge_run-1_space-fsaver...
                magjudge    sub-101_ses-1_task-magjudge_run-2_space-fsaver...
                magjudge    sub-101_ses-1_task-magjudge_run-3_space-fsaver...
                placevalue  sub-101_ses-1_task-placevalue_run-1_space-fsav...
                rest        sub-101_ses-1_task-rest_run-1_space-fsaverage5...
...                                                                       ...
303     1       magjudge    sub-303_ses-1_task-magjudge_run-1_space-fsaver...
                magjudge    sub-303_ses-1_task-magjudge_run-2_space-fsaver...
                magjudge    sub-303_ses-1_task-magjudge_run-3_space-fsaver...
                placevalue  sub-303_ses-1_task-placevalue_run-1_space-fsav...
                rest        sub-303_ses-1_task-rest_run-1_space-fsaverage5...

[224 rows x 1 columns]

In [ ]:
# check 
import json

task = 'digitorder'
task_js = json.load(open(op.join(bids_folder,f'task-{task}_bold.json')))
task_js['RepetitionTime']

## Renaming

In [ ]:
dataset = 'numberline' #'smile' #
bids_folder = op.join(datapool_root_folder, f'ds-{dataset}')
subList = [int(d[4:]) for d in os.listdir(bids_folder) if d.startswith('sub-')]

missingSubs = []
for sub in subList:
    sub_folder = op.join(bids_folder,'derivatives',derivative,f'sub-{sub:03d}')
    fns = [f for f in os.listdir(sub_folder)]
    for fn in fns:
        if '_space-fsaverage5' in fn:
            os.rename(op.join(sub_folder, fn), op.join(sub_folder, fn.replace('_space-fsaverage5', '')))
            print(f'Renamed: {fn}')